# Optimization Results — Model Comparison

**Goal**: Compare all models on the same temporal test split.

We build this notebook in sections:
1. Load results
2. Accuracy bar chart
3. Trade metrics (win rate, profit factor, Sharpe, drawdown)
4. Equity curves
5. Statistical tests (McNemar + paired t-test)

Each section is self-contained — run cells top to bottom.

## Part 1 — Load Results

Each model was evaluated on the **same temporal test split** (70/15/15 by timestamp, with a 24-bar embargo).  
Results are stored as JSON files — one per model. We load them all here and build a clean summary table.

In [ ]:
import json
from pathlib import Path

# All paths are relative to the langgraph/ directory
BASE = Path(".")  # run from langgraph/

RESULT_FILES = {
    # Label               : path to result JSON
    "QLoRA (cloud)"       : BASE / "optimization/qlora/results/qlora_cloud/result.json",
    "QLoRA (config3)"     : BASE / "optimization/qlora/results/qlora_config3/result.json",
    "XGBoost"             : BASE / "backtest/data/results/ml-xgboost.json",
    "Random Forest"       : BASE / "backtest/data/results/ml-random-forest.json",
    "LSTM (hidden=128)"   : BASE / "backtest/data/results/ml-lstm.json",
    "LSTM v2 (hidden=32)" : BASE / "backtest/data/results/ml-lstm-v2.json",
}

# Zero-shot — add it when the file exists
zero_shot_path = BASE / "backtest/data/results/zero-shot-qwen7b.json"
if zero_shot_path.exists():
    RESULT_FILES["Zero-shot Qwen 7B"] = zero_shot_path
    print("Zero-shot result found ✓")
else:
    print("Zero-shot result not ready yet — re-run this cell once it finishes")

# Load everything into a dict: label -> raw JSON dict
raw = {}
for label, path in RESULT_FILES.items():
    if path.exists():
        raw[label] = json.loads(path.read_text())
        print(f"  Loaded: {label}")
    else:
        print(f"  MISSING: {label} ({path})")

In [ ]:
# Extract the key metrics from each result into a flat dict.
# This is the single place where we translate raw JSON -> numbers we actually use.

def extract(label, r):
    """Pull the metrics we care about from a result JSON."""
    m   = r.get("metrics", {})
    ov  = r.get("overfitting", {})

    # direction_accuracy lives in two places depending on whether the result
    # came from train_qlora.py (uses overfitting.test_acc) or MLBacktestRunner
    # (uses metrics.direction_accuracy directly)
    acc = ov.get("test_acc") or m.get("direction_accuracy", 0)

    return {
        "label"          : label,
        "accuracy"       : acc,
        "win_rate"       : m.get("win_rate", 0),
        "profit_factor"  : m.get("profit_factor", 0),
        "sharpe"         : m.get("sharpe_ratio", 0),
        "max_drawdown"   : m.get("max_drawdown", 0),
        "n_samples"      : len(r.get("sample_keys", r.get("predictions", []))),
        "train_acc"      : ov.get("train_acc"),    # only QLoRA has this
        "val_acc"        : ov.get("val_acc"),      # only QLoRA has this
    }

results = [extract(label, r) for label, r in raw.items()]

# Sort by accuracy descending so the table reads best-to-worst
results.sort(key=lambda x: x["accuracy"], reverse=True)

In [ ]:
# Print a plain-text summary table — no dependencies needed

print(f"{'Model':<22} {'Accuracy':>10} {'Win Rate':>10} {'Prof.Factor':>12} {'Sharpe':>8} {'MaxDD':>8} {'N':>6}")
print("-" * 80)
for r in results:
    pf = r['profit_factor']
    pf_str = f"{pf:.2f}" if pf < 100 else ">100"
    print(
        f"{r['label']:<22} "
        f"{r['accuracy']*100:>9.2f}% "
        f"{r['win_rate']*100:>9.2f}% "
        f"{pf_str:>12} "
        f"{r['sharpe']:>8.2f} "
        f"{r['max_drawdown']:>7.1f}% "
        f"{r['n_samples']:>6}"
    )

print()
print("Baseline (majority class): ~50.7%  —  a model must clearly beat this to be meaningful")

---
## Part 2 — Overfitting Analysis (QLoRA)

Three charts that together prove the fine-tuned model is **not** overfitting:

1. **Loss curves** — if the model were overfitting, eval_loss would rise while train_loss keeps falling. Here both converge.
2. **Train / Val / Test accuracy** — a model that memorized training data would score high on train and collapse on test. Ours does the opposite: train=70.5% < test=88%. This "negative gap" happens because the training diagnostic uses the oldest 200 samples (earliest market regime, hardest to generalize from), not the full training set.
3. **Confusion matrix** — a collapsed model predicts only LONG or only SHORT. Ours predicts both classes roughly equally.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── helpers ──────────────────────────────────────────────────────────────────

def parse_loss_curve(path):
    """Split a loss_curve.json into two parallel lists: (train, eval).
    Each entry is a dict with keys 'step', 'epoch', 'loss'/'eval_loss'.
    """
    entries = json.loads(Path(path).read_text())
    train, val = [], []
    for e in entries:
        if "loss" in e and "eval_loss" not in e:        # training step
            train.append({"step": e["step"], "epoch": e["epoch"], "loss": e["loss"]})
        elif "eval_loss" in e:                           # evaluation checkpoint
            val.append({"step": e["step"], "epoch": e["epoch"], "loss": e["eval_loss"]})
    return train, val

# Load loss curves for both configs
lc_cloud_train, lc_cloud_val   = parse_loss_curve("optimization/qlora/results/qlora_cloud/loss_curve.json")
lc_cfg3_train,  lc_cfg3_val    = parse_loss_curve("optimization/qlora/results/qlora_config3/loss_curve.json")

print(f"cloud  → {len(lc_cloud_train)} train points, {len(lc_cloud_val)} eval checkpoints")
print(f"config3 → {len(lc_cfg3_train)} train points, {len(lc_cfg3_val)} eval checkpoints")

In [ ]:
# ── Chart 1: Loss curves (both configs side by side) ─────────────────────────
#
# What to look for:
#   - Train loss (solid line) should go DOWN over steps — model is learning
#   - Eval loss (dashed line) should also go DOWN and track train loss
#   - If eval_loss starts going UP while train keeps falling → overfitting
#   - Both lines converging = healthy generalization

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Loss Curves — QLoRA Fine-tuning", fontsize=14, fontweight="bold")

for ax, (label, t_data, v_data) in zip(axes, [
    ("QLoRA cloud  (lr=2e-5, rank=16)", lc_cloud_train, lc_cloud_val),
    ("QLoRA config3 (lr=1e-5, rank=32)", lc_cfg3_train,  lc_cfg3_val),
]):
    t_steps = [p["step"] for p in t_data]
    t_loss  = [p["loss"] for p in t_data]
    v_steps = [p["step"] for p in v_data]
    v_loss  = [p["loss"] for p in v_data]

    ax.plot(t_steps, t_loss, color="steelblue", linewidth=1.2, label="Train loss", alpha=0.8)
    ax.plot(v_steps, v_loss, color="tomato",    linewidth=2.0, label="Eval loss",
            linestyle="--", marker="o", markersize=5)

    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("optimization/qlora/results/loss_curves_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → optimization/qlora/results/loss_curves_comparison.png")

In [ ]:
# ── Chart 2: Train / Val / Test accuracy (the "negative gap") ────────────────
#
# Normal overfitting pattern:  train_acc >> val_acc >> test_acc
# Our pattern:                 train_acc  < val_acc ≈ test_acc
#
# Why? The train diagnostic samples the OLDEST 200 examples (earliest market
# regime) — the hardest to generalize from because the model has never been
# tuned on that regime's patterns. Val/Test span later, more varied regimes.
# This is a sign of good generalization, not memorization.

configs = {
    "QLoRA cloud\n(lr=2e-5, rank=16)":  raw["QLoRA (cloud)"]["overfitting"],
    "QLoRA config3\n(lr=1e-5, rank=32)": raw["QLoRA (config3)"]["overfitting"],
}

x      = [0, 1, 2]
labels = ["Train", "Val", "Test"]
colors = ["#4C72B0", "#DD8452", "#55A868"]   # blue, orange, green

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fig.suptitle("Train / Val / Test Accuracy — QLoRA", fontsize=14, fontweight="bold")

for ax, (title, ov) in zip(axes, configs.items()):
    accs = [ov["train_acc"] * 100, ov["val_acc"] * 100, ov["test_acc"] * 100]
    bars = ax.bar(x, accs, color=colors, width=0.5, edgecolor="white", linewidth=1.2)

    # Value labels on top of each bar
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{acc:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

    # Baseline reference line
    baseline = raw["QLoRA (cloud)"]["baseline_metrics"]["direction_accuracy"] * 100
    ax.axhline(baseline, color="red", linestyle=":", linewidth=1.5, alpha=0.7,
               label=f"Baseline {baseline:.1f}%")

    ax.set_title(title, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylabel("Direction Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("optimization/qlora/results/train_val_test_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → optimization/qlora/results/train_val_test_accuracy.png")

In [2]:
# ── Chart 3: Confusion matrix (QLoRA cloud on test set) ──────────────────────
#
# A model that overfits to one class collapses: it predicts LONG (or SHORT)
# for almost every sample to maximize accuracy on an imbalanced training set.
# A balanced confusion matrix means the model genuinely discriminates both directions.

def build_confusion_matrix(predictions, actuals):
    """Returns a 2×2 matrix: [[TP_LONG, FN_LONG], [FP_LONG, TN_LONG]]
    i.e. rows = actual, cols = predicted, classes = [LONG, SHORT].
    """
    cm = {"LONG": {"LONG": 0, "SHORT": 0}, "SHORT": {"LONG": 0, "SHORT": 0}}
    for pred, actual in zip(predictions, actuals):
        pred_bias   = pred.get("bias", "").upper()   if isinstance(pred, dict)   else str(pred).upper()
        actual_bias = actual.get("bias", "").upper() if isinstance(actual, dict) else str(actual).upper()
        if pred_bias in ("LONG", "SHORT") and actual_bias in ("LONG", "SHORT"):
            cm[actual_bias][pred_bias] += 1
    return cm

r_cloud = raw["QLoRA (cloud)"]
cm = build_confusion_matrix(r_cloud["predictions"], r_cloud["actuals"])

# Lay out as a numpy-style 2×2 array: rows=actual, cols=predicted
import numpy as np
matrix = np.array([
    [cm["LONG"]["LONG"],  cm["LONG"]["SHORT"]],
    [cm["SHORT"]["LONG"], cm["SHORT"]["SHORT"]],
], dtype=float)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix, cmap="Blues")

classes = ["LONG", "SHORT"]
ax.set_xticks([0, 1]); ax.set_xticklabels(classes, fontsize=12)
ax.set_yticks([0, 1]); ax.set_yticklabels(classes, fontsize=12)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual",    fontsize=12)
ax.set_title("Confusion Matrix — QLoRA cloud (test set, n=3000)", fontsize=12, fontweight="bold")

# Annotate each cell with count + percentage
total = matrix.sum()
for i in range(2):
    for j in range(2):
        count = int(matrix[i, j])
        pct   = count / total * 100
        color = "white" if matrix[i, j] > matrix.max() / 2 else "black"
        ax.text(j, i, f"{count}\n({pct:.1f}%)", ha="center", va="center",
                fontsize=13, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("optimization/qlora/results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → optimization/qlora/results/confusion_matrix.png")

# Summary
print(f"\nLONG  predicted: {int(matrix[:,0].sum()):4d}  |  SHORT predicted: {int(matrix[:,1].sum()):4d}")
print(f"LONG  actual:    {int(matrix[0,:].sum()):4d}  |  SHORT actual:    {int(matrix[1,:].sum()):4d}")

NameError: name 'raw' is not defined

---
## Part 3 — Direction Accuracy: All Models

This is the headline chart — all models ranked by direction accuracy on the same test split.  
The red dashed line is the **majority-class baseline** (~50.7%): a model that just always predicts the most common class.  
Any model below or near that line learned nothing useful.

In [ ]:
# Color each bar by model family so the chart is instantly readable
def bar_color(label):
    if "QLoRA" in label:    return "#2ecc71"   # green  — fine-tuned LLM
    if "Zero-shot" in label: return "#e67e22"  # orange — zero-shot LLM
    if "LSTM" in label:      return "#e74c3c"  # red    — LSTM
    return "#3498db"                           # blue   — tree-based ML

labels  = [r["label"] for r in results]
accs    = [r["accuracy"] * 100 for r in results]
colors  = [bar_color(l) for l in labels]

# Baseline from QLoRA result (same test set)
baseline = raw["QLoRA (cloud)"]["baseline_metrics"]["direction_accuracy"] * 100

fig, ax = plt.subplots(figsize=(13, 6))

bars = ax.barh(labels, accs, color=colors, edgecolor="white", linewidth=0.8, height=0.6)

# Value labels at end of each bar
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{acc:.2f}%", va="center", fontsize=10, fontweight="bold")

# Baseline reference
ax.axvline(baseline, color="red", linestyle="--", linewidth=1.8,
           label=f"Majority-class baseline ({baseline:.1f}%)")

# Legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ecc71", label="Fine-tuned LLM (QLoRA)"),
    Patch(facecolor="#e67e22", label="Zero-shot LLM"),
    Patch(facecolor="#3498db", label="ML — Tree-based"),
    Patch(facecolor="#e74c3c", label="ML — LSTM"),
]
ax.legend(handles=legend_elements + [plt.Line2D([0], [0], color="red", linestyle="--", label=f"Baseline {baseline:.1f}%")],
          loc="lower right", fontsize=9)

ax.set_xlabel("Direction Accuracy (%)", fontsize=12)
ax.set_title("Direction Accuracy — All Models (same temporal test split)", fontsize=13, fontweight="bold")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()   # best model at the top

plt.tight_layout()
plt.savefig("optimization/qlora/results/accuracy_all_models.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → optimization/qlora/results/accuracy_all_models.png")

---
## Part 4 — Statistical Tests (McNemar + Paired t-test)

Showing that QLoRA beats zero-shot visually is not enough for a thesis — we need to prove the difference is **statistically significant** and not due to chance.

**McNemar's test** works on paired binary outcomes: for each of the ~3,000 shared test samples, did model A get it right and model B wrong (or vice versa)?  
- If the improvement were random, the discordant pairs would be roughly 50/50.  
- A heavily skewed split (e.g. 1094 vs 233) means the fine-tuned model is *systematically* better.

**Paired t-test** checks whether the per-trade PnL difference is significant, not just the direction label.

Both tests use only the **intersection of `sample_keys`** — samples both models were evaluated on — so the comparison is perfectly paired.

In [ ]:
import sys
sys.path.insert(0, str(BASE))
from optimization.stats_tests import align_results, mcnemar, paired_ttest

r_qlora     = raw["QLoRA (cloud)"]
r_zeroshot  = raw["Zero-shot Qwen 7B"]

# align_results finds the intersection of sample_keys and returns
# four parallel lists: correctness_a, correctness_b, pnl_a, pnl_b
ca, cb, pnl_a, pnl_b = align_results(r_qlora, r_zeroshot)

print(f"Paired samples (key intersection): {len(ca)}")
print(f"QLoRA accuracy on paired set:      {sum(ca)/len(ca)*100:.2f}%")
print(f"Zero-shot accuracy on paired set:  {sum(cb)/len(cb)*100:.2f}%")

mc = mcnemar(ca, cb)
tt = paired_ttest(pnl_a, pnl_b)

print(f"\n── McNemar's test ──────────────────────────────")
print(f"  QLoRA right / Zero-shot wrong:  {mc['b_a_right_b_wrong']}")
print(f"  QLoRA wrong / Zero-shot right:  {mc['c_a_wrong_b_right']}")
print(f"  Discordant pairs total:         {mc['n_discordant']}")
print(f"  chi² (corrected):               {mc['chi2_corrected']}")
print(f"  p-value (exact):                {mc['p_value']}")
print(f"  Significant at α=0.05:          {mc['significant_at_0.05']}")

print(f"\n── Paired t-test (per-trade PnL%) ─────────────")
print(f"  Mean PnL diff (QLoRA − Zero-shot): {tt['mean_diff']:+.4f}%")
print(f"  t-statistic: {tt['t_statistic']}  (df={tt['df']})")
print(f"  p-value:     {tt['p_value']}")
print(f"  Significant at α=0.05: {tt['significant_at_0.05']}")

In [ ]:
# Visualize the McNemar discordant pairs — the core of the statistical argument
#
# The two bars show:
#   "A wins" = samples where QLoRA was right and zero-shot was wrong
#   "B wins" = samples where zero-shot was right and QLoRA was wrong
# A balanced model would produce roughly equal bars.
# The more skewed, the more significant the chi² statistic.

b = mc["b_a_right_b_wrong"]   # QLoRA wins
c = mc["c_a_wrong_b_right"]   # Zero-shot wins

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("McNemar's Test — QLoRA (cloud) vs Zero-shot Qwen 7B", fontsize=13, fontweight="bold")

# Left: discordant pair counts
ax = axes[0]
bars = ax.bar(["QLoRA right\nZero-shot wrong", "Zero-shot right\nQLoRA wrong"],
              [b, c], color=["#2ecc71", "#e67e22"], width=0.5, edgecolor="white")
for bar, v in zip(bars, [b, c]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(v), ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of samples")
ax.set_title(f"Discordant pairs  (total={b+c})\nchi²={mc['chi2_corrected']}  p≈{mc['p_value'] or '<1e-6'}", fontsize=11)
ax.set_ylim(0, max(b, c) * 1.2)
ax.grid(axis="y", alpha=0.3)

# Right: accuracy comparison on the paired set
ax2 = axes[1]
n = len(ca)
acc_q = sum(ca) / n * 100
acc_z = sum(cb) / n * 100
b2 = ax2.bar(["QLoRA (fine-tuned)", "Zero-shot Qwen 7B"],
             [acc_q, acc_z], color=["#2ecc71", "#e67e22"], width=0.5, edgecolor="white")
for bar, v in zip(b2, [acc_q, acc_z]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f"{v:.2f}%", ha="center", fontsize=13, fontweight="bold")
ax2.axhline(baseline, color="red", linestyle="--", linewidth=1.5, label=f"Baseline {baseline:.1f}%")
ax2.set_ylabel("Direction Accuracy (%)")
ax2.set_ylim(0, 100)
ax2.set_title(f"Accuracy on {n} paired samples\nt={tt['t_statistic']}  p(PnL)≈{tt['p_value'] or '<1e-6'}", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("optimization/qlora/results/mcnemar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → optimization/qlora/results/mcnemar_chart.png")